In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from collections import Counter
import math

In [3]:
df = pd.read_parquet("20240611.parquet")

print(df.shape)
df.head()

(728592, 37)


,DST_IP,DST_IP_SUBNET,DST_IP_VERSION,DST_ASN,DST_COUNTRY,DST_PORT,PROTOCOL,TIME_FIRST,TIME_LAST,DURATION,...,QUIC_TLS_EXT_TYPE,QUIC_PACKETS,PPI,PPI_LEN,PPI_DURATION,PPI_ROUNDTRIPS,PHIST_SRC_SIZES,PHIST_DST_SIZES,PHIST_SRC_IPT,PHIST_DST_IPT
0,9605b73dfd,ae12b508c7,4,15169,US,443,17,2024-06-10 23:00:00+02:00,2024-06-10 23:02:06.182160+02:00,126.182160,...,"[27, 65037, 16, 13, 45, 57, 17513, 43, 0, 10, ...","[129, 130, 133, 128, 128, 128, 133, 132, 128, ...","[[0, 2, 13, 0, 0, 13, 0, 0, 1, 0, 4, 0, 0, 0, ...",30,0.110,6,"[0, 24, 52, 3, 3, 3, 4, 16]","[0, 32, 0, 7, 4, 8, 7, 125]","[78, 9, 3, 8, 1, 0, 0, 5]","[158, 3, 7, 8, 0, 1, 0, 5]"
1,8be224584e,d6c24afba9,4,15169,US,443,17,2024-06-10 23:00:00+02:00,2024-06-10 23:04:58.284946+02:00,298.284946,...,"[17513, 10, 16, 42, 45, 57, 0, 27, 43, 51, 13,...","[129, 130, 133, 132, 132, 132, 132, 132, 132, ...","[[0, 3, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0,...",30,0.011,3,"[0, 0, 22626, 2, 1, 0, 0, 84]","[0, 254, 154, 70, 15, 59, 82, 210869]","[22468, 173, 9, 1, 0, 0, 0, 61]","[211401, 35, 4, 1, 0, 0, 0, 61]"
2,fb20b00904,8737b0d8c0,6,15169,IE,443,17,2024-06-10 23:00:00+02:00,2024-06-10 23:00:32.069013+02:00,32.069013,...,"[45, 27, 10, 16, 43, 57, 0, 51, 17513, 13]","[129, 133, 132, 132, 132, 132, 132, 132, 132, ...","[[0, 2, 0, 0, 0, 0, 4, 1, 1, 2, 0, 0, 1, 0, 0,...",30,1.248,6,"[0, 0, 189, 2, 1, 3, 4, 4]","[0, 15, 0, 4, 3, 1, 3, 2380]","[197, 1, 0, 0, 0, 1, 0, 3]","[2400, 1, 0, 0, 0, 1, 0, 3]"
3,1e076397c8,42525e1ae0,6,396982,US,443,17,2024-06-10 23:00:00+02:00,2024-06-10 23:00:00.098561+02:00,0.098561,...,"[65037, 57, 10, 51, 27, 17513, 16, 13, 45, 0, ...","[129, 133, 132, 128, 128, 128, 128, 128, 128, ...","[[0, 15, 9, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0...",23,0.099,6,"[0, 1, 2, 0, 1, 0, 1, 8]","[0, 6, 0, 1, 1, 0, 1, 1]","[10, 2, 0, 0, 0, 0, 0, 0]","[7, 1, 1, 0, 0, 0, 0, 0]"
4,0e361e0920,9962760d94,4,15169,US,443,17,2024-06-10 23:00:00+02:00,2024-06-10 23:00:00.141653+02:00,0.141653,...,"[27, 51, 13, 17513, 0, 65037, 57, 10, 43, 16, 45]","[129, 129, 129, 132, 132, 132, 132, 132, 132, ...","[[0, 1, 13, 1, 0, 0, 0, 11, 1, 6, 1, 0, 23, 1,...",28,0.142,7,"[0, 1, 9, 2, 0, 1, 0, 3]","[0, 3, 1, 1, 1, 1, 1, 4]","[13, 1, 1, 0, 0, 0, 0, 0]","[7, 2, 2, 0, 0, 0, 0, 0]"


In [4]:
ppi_features = df[
    [
        "PPI",
        "PPI_LEN",
        "PPI_DURATION",
        "PPI_ROUNDTRIPS"
    ]
].copy()

ppi_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 728592 entries, 0 to 728591
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PPI             728592 non-null  object 
 1   PPI_LEN         728592 non-null  int64  
 2   PPI_DURATION    728592 non-null  float64
 3   PPI_ROUNDTRIPS  728592 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 22.2+ MB


In [5]:
# Calculate the average inter-packet time for each flow
ppi_features["mean_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[0])
)

In [6]:
# Calculate the variability of inter-packet times
ppi_features["std_ipt"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[0])
)

In [7]:
# Calculate the average packet size
ppi_features["mean_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.mean(x[2])
)

In [8]:
# Calculate packet size variability
ppi_features["std_packet_size"] = ppi_features["PPI"].apply(
    lambda x: np.std(x[2])
)

In [9]:
def direction_change_ratio(directions):

    # If there is only one packet, no direction change is possible
    if len(directions) < 2:
        return 0

    changes = 0

    # Compare every packet direction with the next one
    for i in range(len(directions) - 1):

        if directions[i] != directions[i + 1]:
            changes += 1

    # Return the proportion of direction changes
    return changes / (len(directions) - 1)

In [10]:
ppi_features["direction_change_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: direction_change_ratio(x[1])
    )
)

In [11]:
def forward_packet_ratio(directions):

    if len(directions) == 0:
        return 0

    forward_packets = np.sum(directions == 1)

    return forward_packets / len(directions)

In [12]:
ppi_features["forward_packet_ratio"] = (
    ppi_features["PPI"].apply(
        lambda x: forward_packet_ratio(x[1])
    )
)

In [13]:
ppi_features["log_ppi_duration"] = np.log1p(ppi_features["PPI_DURATION"])
ppi_features["log_mean_ipt"] = np.log1p(ppi_features["mean_ipt"])
ppi_features["log_std_ipt"] = np.log1p(ppi_features["std_ipt"])

In [14]:
ppi_final = ppi_features[
    [
        # Original Features
        "PPI_LEN",
        "PPI_ROUNDTRIPS",

        # Log-transformed Features
        "log_ppi_duration",
        "log_mean_ipt",
        "log_std_ipt",

        # Engineered Features
        "mean_packet_size",
        "std_packet_size",
        "direction_change_ratio",
        "forward_packet_ratio"
    ]
].copy()

print("Final PPI Feature Bank Shape:", ppi_final.shape)

ppi_final.head()

Final PPI Feature Bank Shape: (728592, 9)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio
0,30,6,0.104360,1.540445,2.011873,435.633333,504.944649,0.413793,0.466667
1,30,3,0.010940,0.312375,0.717000,629.766667,557.518351,0.172414,0.266667
2,30,6,0.810041,3.751854,5.377717,488.966667,517.350267,0.379310,0.433333
3,23,6,0.094401,1.668527,2.072350,568.173913,557.311220,0.500000,0.565217
4,28,7,0.132781,1.803594,2.261974,408.500000,520.425478,0.481481,0.571429


In [15]:
cid_features = df[
    [
        "DST_ASN",
        "QUIC_OCCID",
        "QUIC_OSCID",
        "QUIC_SCID",
        "QUIC_RETRY_SCID",
        "QUIC_SNI"
    ]
].copy()

cid_features.head()

,DST_ASN,QUIC_OCCID,QUIC_OSCID,QUIC_SCID,QUIC_RETRY_SCID,QUIC_SNI
0,15169,,305d21d6ec7b9b57,f05d21d6ec7b9b57,,play-fe.googleapis.com
1,15169,,d924282caed6dc7d,f924282caed6dc7d,,rr1---sn-2gb7sn7y.googlevideo.com
2,15169,,8f648c8f60bb2dc6,ef648c8f60bb2dc6,,rr3---sn-2gb7snez.googlevideo.com
3,396982,,dd5a419de6c3799f,fd5a419de6c3799f,,spclient.wg.spotify.com
4,15169,,e87030589f981187,e87030589f981187,,lamssettings-pa.googleapis.com


In [16]:
cid_features["occid_length"] = cid_features["QUIC_OCCID"].str.len()

cid_features["oscid_length"] = cid_features["QUIC_OSCID"].str.len()

cid_features["scid_length"] = cid_features["QUIC_SCID"].str.len()

cid_features["retry_length"] = cid_features["QUIC_RETRY_SCID"].str.len()

cid_features["sni_length"] = cid_features["QUIC_SNI"].str.len()

cid_features["sni_labels"] = (
    cid_features["QUIC_SNI"]
    .str.split(".")
    .str.len()
)

In [17]:
cid_features["retry_present"] = (cid_features["retry_length"] > 0).astype(int)

In [18]:
def shannon_entropy(text):

    if len(text) == 0:
        return 0.0

    counts = Counter(text)

    entropy = 0.0

    length = len(text)

    for count in counts.values():

        p = count / length

        entropy -= p * math.log2(p)

    return entropy

In [19]:
def normalized_entropy(text):

    if len(text) == 0:
        return 0.0

    H = shannon_entropy(text)

    Hmax = math.log2(min(len(text), 16))

    return H / Hmax if Hmax > 0 else 0.0

In [20]:
cid_features["occid_entropy"] = (cid_features["QUIC_OCCID"].apply(normalized_entropy))

cid_features["oscid_entropy"] = (cid_features["QUIC_OSCID"].apply(normalized_entropy))

cid_features["scid_entropy"] = (cid_features["QUIC_SCID"].apply(normalized_entropy))

In [21]:
cid_features[
    [
        "occid_entropy",
        "oscid_entropy",
        "scid_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
occid_entropy,728592.0,0.237969,0.396702,0.000000,0.00000,0.000000,0.742098,1.000000
oscid_entropy,728592.0,0.815849,0.059549,0.480295,0.78125,0.812500,0.863205,0.988492
scid_entropy,728592.0,0.769065,0.174915,0.000000,0.75766,0.800705,0.843750,1.000000


In [22]:
MIN_FLOWS = max(50, int(0.001 * len(cid_features)))

In [23]:
asn_groups = cid_features.groupby("DST_ASN")

In [24]:
asn_profile = asn_groups.agg({

    "scid_length": ["count", "median"],

    "scid_entropy": "median",

    "oscid_length": "median",

    "oscid_entropy": "median",

    "retry_present": "mean",

    "sni_length": "median",

    "sni_labels": "median"

})

In [25]:
asn_profile.columns = [

    "flows",
    "median_scid_length",
    "median_scid_entropy",
    "median_oscid_length",
    "median_oscid_entropy",
    "retry_rate",
    "median_sni_length",
    "median_sni_labels",

]

In [26]:
asn_profile = asn_profile[asn_profile["flows"] >= MIN_FLOWS]

In [27]:
asn_profile.head()

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
6185,1051,40.0,0.925925,16.0,0.800705,0.000000,17.0,3.0
8075,9709,28.0,0.883896,16.0,0.800705,0.000206,21.0,3.0
13335,14911,40.0,0.904613,16.0,0.820160,0.000000,16.0,3.0
15169,559934,16.0,0.800705,16.0,0.812500,0.000000,20.0,3.0
16509,3698,40.0,0.921832,16.0,0.823109,0.000000,18.0,3.0


In [28]:
cid_features = cid_features.merge(
    asn_profile,
    on="DST_ASN",
    how="left"
)

In [29]:
cid_features["known_asn"] = (
    cid_features["flows"].notna().astype(int)
)

In [30]:
global_scid_length = cid_features["scid_length"].median()

global_scid_entropy = cid_features["scid_entropy"].median()

global_oscid_length = cid_features["oscid_length"].median()

global_oscid_entropy = cid_features["oscid_entropy"].median()

global_retry_rate = cid_features["retry_present"].mean()

global_sni_length = cid_features["sni_length"].median()

global_sni_labels = cid_features["sni_labels"].median()

In [31]:
fill_values = {
    "median_scid_length": global_scid_length,
    "median_scid_entropy": global_scid_entropy,
    "median_oscid_length": global_oscid_length,
    "median_oscid_entropy": global_oscid_entropy,
    "retry_rate": global_retry_rate,
    "median_sni_length": global_sni_length,
    "median_sni_labels": global_sni_labels
}

cid_features = cid_features.fillna(fill_values)

In [32]:
asn_profile.describe().T

,count,mean,std,min,25%,50%,75%,max
flows,12.0,60527.250000,158756.554196,1051.000000,3091.500000,5707.500000,23608.000000,559934.000000
median_scid_length,12.0,28.500000,11.571910,16.000000,16.000000,31.000000,40.000000,40.000000
median_scid_entropy,12.0,0.863387,0.064535,0.769455,0.800705,0.894255,0.922855,0.926550
median_oscid_length,12.0,16.000000,0.000000,16.000000,16.000000,16.000000,16.000000,16.000000
median_oscid_entropy,12.0,0.814585,0.010887,0.800705,0.809551,0.812500,0.820897,0.831955
retry_rate,12.0,0.000022,0.000060,0.000000,0.000000,0.000000,0.000000,0.000206
median_sni_length,12.0,20.416667,5.884623,13.000000,16.750000,19.000000,22.000000,33.000000
median_sni_labels,12.0,3.083333,0.288675,3.000000,3.000000,3.000000,3.000000,4.000000


In [33]:
asn_profile.sort_values("flows", ascending=False).head(20)

,flows,median_scid_length,median_scid_entropy,median_oscid_length,median_oscid_entropy,retry_rate,median_sni_length,median_sni_labels
DST_ASN,,,,,,,,
15169,559934,16.0,0.800705,16.0,0.812500,0.000000,20.0,3.0
32934,69179,16.0,0.769455,16.0,0.812500,0.000000,18.0,3.0
396982,49699,16.0,0.800705,16.0,0.812500,0.000060,29.0,3.0
13335,14911,40.0,0.904613,16.0,0.820160,0.000000,16.0,3.0
8075,9709,28.0,0.883896,16.0,0.800705,0.000206,21.0,3.0
54113,7542,34.0,0.911428,16.0,0.815733,0.000000,20.0,4.0
20940,3873,16.0,0.788910,16.0,0.812500,0.000000,25.0,3.0
16509,3698,40.0,0.921832,16.0,0.823109,0.000000,18.0,3.0
36183,3487,40.0,0.926550,16.0,0.831955,0.000000,15.0,3.0


In [34]:
cid_features["scid_length_deviation"] = (
    cid_features["scid_length"] -
    cid_features["median_scid_length"]
).abs()

cid_features["oscid_length_deviation"] = (
    cid_features["oscid_length"] -
    cid_features["median_oscid_length"]
).abs()

In [35]:
cid_features["sni_length_deviation"] = (
    cid_features["sni_length"] -
    cid_features["median_sni_length"]
).abs()

In [36]:
cid_features["scid_entropy_deviation"] = (
    cid_features["scid_entropy"] -
    cid_features["median_scid_entropy"]
).abs()

cid_features["oscid_entropy_deviation"] = (
    cid_features["oscid_entropy"] -
    cid_features["median_oscid_entropy"]
).abs()

In [37]:
cid_final = cid_features[
    [
        # Raw CID Features
        "occid_length",
        "oscid_length",
        "scid_length",
        "retry_present",
        "sni_length",
        "sni_labels",

        # Statistical Features
        "oscid_entropy",
        "scid_entropy",

        # Context Features
        "scid_length_deviation",
        "oscid_length_deviation",
        "scid_entropy_deviation",
        "oscid_entropy_deviation",
        "sni_length_deviation",
        "known_asn"
    ]
].copy()


print(cid_final.shape)

(728592, 14)


In [38]:
cid_final.head()

,occid_length,oscid_length,scid_length,retry_present,sni_length,sni_labels,oscid_entropy,scid_entropy,scid_length_deviation,oscid_length_deviation,scid_entropy_deviation,oscid_entropy_deviation,sni_length_deviation,known_asn
0,0,16,16,0,22,3,0.875000,0.875000,0.0,0.0,0.074295,0.062500,2.0,1
1,0,16,16,0,33,3,0.769455,0.820160,0.0,0.0,0.019455,0.043045,13.0,1
2,0,16,16,0,33,3,0.757660,0.800705,0.0,0.0,0.000000,0.054840,13.0,1
3,0,16,16,0,23,4,0.851410,0.863205,0.0,0.0,0.062500,0.038910,6.0,1
4,0,16,16,0,30,3,0.750000,0.750000,0.0,0.0,0.050705,0.062500,10.0,1


In [39]:
hist_features = df[
    [
        "PHIST_SRC_SIZES",
        "PHIST_DST_SIZES",
        "PHIST_SRC_IPT",
        "PHIST_DST_IPT"
    ]
].copy()

In [40]:
def histogram_entropy(hist):

    # Convert to NumPy array
    hist = np.array(hist, dtype=float)

    # Total observations
    total = hist.sum()

    # Handle empty histograms
    if total == 0:
        return 0

    # Convert counts to probabilities
    probabilities = hist / total

    # Calculate Shannon entropy
    entropy = 0

    for p in probabilities:
        if p > 0:
            entropy -= p * math.log2(p)

    # Normalize entropy
    max_entropy = math.log2(len(hist))

    return entropy / max_entropy

In [41]:
# Source packet size histogram entropy
hist_features["src_size_entropy"] = (
    hist_features["PHIST_SRC_SIZES"]
    .apply(histogram_entropy)
)

# Destination packet size histogram entropy
hist_features["dst_size_entropy"] = (
    hist_features["PHIST_DST_SIZES"]
    .apply(histogram_entropy)
)

# Source IPT histogram entropy
hist_features["src_ipt_entropy"] = (
    hist_features["PHIST_SRC_IPT"]
    .apply(histogram_entropy)
)

# Destination IPT histogram entropy
hist_features["dst_ipt_entropy"] = (
    hist_features["PHIST_DST_IPT"]
    .apply(histogram_entropy)
)

In [42]:
hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
src_size_entropy,728592.0,0.567640,0.182613,0.0,0.507214,0.608376,0.689010,0.917811
dst_size_entropy,728592.0,0.576660,0.205130,0.0,0.514751,0.636533,0.708605,0.935785
src_ipt_entropy,728592.0,0.350827,0.195695,0.0,0.226455,0.333333,0.486383,1.000000
dst_ipt_entropy,728592.0,0.350185,0.216324,0.0,0.197224,0.346307,0.507309,1.000000


In [43]:
hist_final = hist_features[
    [
        "src_size_entropy",
        "dst_size_entropy",
        "src_ipt_entropy",
        "dst_ipt_entropy"
    ]
].copy()

print(hist_final.shape)

hist_final.head()

(728592, 4)


,src_size_entropy,dst_size_entropy,src_ipt_entropy,dst_ipt_entropy
0,0.673870,0.497900,0.441316,0.279131
1,0.012406,0.012017,0.032356,0.002180
2,0.170845,0.035084,0.067095,0.008123
3,0.566813,0.590317,0.216674,0.328809
4,0.598246,0.840685,0.233281,0.436432


In [44]:
flow_df = df.copy()

In [45]:
flow_df["duration_safe"] = flow_df["DURATION"].clip(lower=1e-6)

In [46]:
flow_df["total_bytes"] = (
    flow_df["BYTES"] +
    flow_df["BYTES_REV"]
)

In [47]:
flow_df["total_packets"] = (
    flow_df["PACKETS"] +
    flow_df["PACKETS_REV"]
)

In [48]:
flow_df["byte_rate"] = (
    flow_df["total_bytes"] /
    flow_df["duration_safe"]
)

In [49]:
flow_df["packet_rate"] = (
    flow_df["total_packets"] /
    flow_df["duration_safe"]
)

In [50]:
flow_df["avg_packet_size"] = (
    flow_df["total_bytes"] /
    flow_df["total_packets"].clip(lower=1)
)

In [51]:
flow_df["byte_ratio"] = (
    flow_df["BYTES"] /
    flow_df["BYTES_REV"].clip(lower=1)
)

In [52]:
flow_df["packet_ratio"] = (
    flow_df["PACKETS"] /
    flow_df["PACKETS_REV"].clip(lower=1)
)

In [53]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
]

In [54]:
flow_features.replace([np.inf, -np.inf], np.nan, inplace=True)

flow_features.isnull().sum()

DURATION           0
FLOW_END_REASON    0
total_bytes        0
total_packets      0
byte_rate          0
packet_rate        0
avg_packet_size    0
byte_ratio         0
packet_ratio       0
dtype: int64

In [55]:
flow_features = flow_df[
    [
        "DURATION",
        "FLOW_END_REASON",
        "total_bytes",
        "total_packets",
        "byte_rate",
        "packet_rate",
        "avg_packet_size",
        "byte_ratio",
        "packet_ratio"
    ]
].copy()

In [56]:
flow_features["log_duration"] = np.log1p(flow_features["DURATION"])

flow_features["log_total_bytes"] = np.log1p(flow_features["total_bytes"])

flow_features["log_total_packets"] = np.log1p(flow_features["total_packets"])

flow_features["log_byte_rate"] = np.log1p(flow_features["byte_rate"])

flow_features["log_packet_rate"] = np.log1p(flow_features["packet_rate"])

flow_features["log_byte_ratio"] = np.log1p(flow_features["byte_ratio"])

flow_features["log_packet_ratio"] = np.log1p(flow_features["packet_ratio"])

In [57]:
log_columns = [

"log_duration",

"FLOW_END_REASON",

"log_total_bytes",

"log_total_packets",

"log_byte_rate",

"log_packet_rate",

"avg_packet_size",

"log_byte_ratio",

"log_packet_ratio"

]

flow_features[log_columns].corr()

,log_duration,FLOW_END_REASON,log_total_bytes,log_total_packets,log_byte_rate,log_packet_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
log_duration,1.000000,0.031433,0.525939,0.592627,-0.783301,-0.811770,0.026137,-0.047664,-0.184852
FLOW_END_REASON,0.031433,1.000000,0.000803,0.009971,-0.025485,-0.020104,-0.023957,0.010356,0.015664
log_total_bytes,0.525939,0.000803,1.000000,0.964714,-0.069972,-0.170465,0.466685,-0.276916,-0.417914
log_total_packets,0.592627,0.009971,0.964714,1.000000,-0.190990,-0.249837,0.235056,-0.235807,-0.333585
log_byte_rate,-0.783301,-0.025485,-0.069972,-0.190990,1.000000,0.980182,0.372667,-0.008080,-0.068117
log_packet_rate,-0.811770,-0.020104,-0.170465,-0.249837,0.980182,1.000000,0.207978,0.034440,0.010560
avg_packet_size,0.026137,-0.023957,0.466685,0.235056,0.372667,0.207978,1.000000,-0.196279,-0.463057
log_byte_ratio,-0.047664,0.010356,-0.276916,-0.235807,-0.008080,0.034440,-0.196279,1.000000,0.609338
log_packet_ratio,-0.184852,0.015664,-0.417914,-0.333585,-0.068117,0.010560,-0.463057,0.609338,1.000000


In [58]:
selected_columns = [

    "log_duration",
    "FLOW_END_REASON",
    "log_total_bytes",
    "log_byte_rate",
    "avg_packet_size",
    "log_byte_ratio",
    "log_packet_ratio"

]

flow_features_selected = flow_features[selected_columns].copy()

flow_features_selected.head()

,log_duration,FLOW_END_REASON,log_total_bytes,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio
0,4.845620,1,12.214125,7.377019,700.055556,0.158156,0.453474
1,5.701396,2,19.418273,13.720225,1157.797076,0.005772,0.102005
2,3.498597,1,14.937707,11.469826,1177.305864,0.008245,0.081001
3,0.094001,1,9.559094,11.876110,616.173913,1.671815,0.832909
4,0.132477,1,9.411075,11.365379,436.500000,0.546246,0.847298


In [59]:
quic_features = df[
    [
        "QUIC_VERSION",
        "QUIC_CLIENT_VERSION",
        "QUIC_TOKEN_LENGTH",
        "QUIC_ZERO_RTT",
        "QUIC_MULTIPLEXED"
    ]
].copy()

quic_features.head()

,QUIC_VERSION,QUIC_CLIENT_VERSION,QUIC_TOKEN_LENGTH,QUIC_ZERO_RTT,QUIC_MULTIPLEXED
0,1,1,70,1,0
1,1,1,65,1,0
2,1,1,0,0,0
3,1,1,70,0,0
4,1,1,0,0,0


In [60]:
quic_features["token_present"] = (
    quic_features["QUIC_TOKEN_LENGTH"] > 0
).astype(int)

In [61]:
quic_features["log_token_length"] = np.log1p(
    quic_features["QUIC_TOKEN_LENGTH"]
)

In [62]:
quic_features["zero_rtt_present"] = (
    quic_features["QUIC_ZERO_RTT"] > 0
).astype(int)

In [63]:
quic_features["multiplexed"] = (
    quic_features["QUIC_MULTIPLEXED"] > 0
).astype(int)

In [64]:
version_map = {
    version: idx
    for idx, version in enumerate(
        sorted(quic_features["QUIC_VERSION"].unique())
    )
}

quic_features["quic_version_id"] = (
    quic_features["QUIC_VERSION"].map(version_map)
)

In [65]:
version_map

{np.int64(0): 0,
 np.int64(1): 1,
 np.int64(3467641594): 2,
 np.int64(4207849474): 3,
 np.int64(4207849486): 4,
 np.int64(4207849491): 5,
 np.int64(4278190109): 6}

In [66]:
quic_features["log_zero_rtt"] = np.log1p(
    quic_features["QUIC_ZERO_RTT"]
)

In [67]:
quic_features["log_multiplexed"] = np.log1p(
    quic_features["QUIC_MULTIPLEXED"]
)

In [68]:
quic_selected = quic_features[
    [
        "quic_version_id",
        "log_token_length",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
]

quic_selected.corr()

,quic_version_id,log_token_length,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
quic_version_id,1.000000,0.118899,-0.068577,-0.080799,-0.044561,-0.046832
log_token_length,0.118899,1.000000,0.646665,0.728737,-0.010489,-0.012739
log_zero_rtt,-0.068577,0.646665,1.000000,0.887696,-0.023816,-0.046543
zero_rtt_present,-0.080799,0.728737,0.887696,1.000000,-0.060684,-0.074775
log_multiplexed,-0.044561,-0.010489,-0.023816,-0.060684,1.000000,0.949987
multiplexed,-0.046832,-0.012739,-0.046543,-0.074775,0.949987,1.000000


In [69]:
selected_quic = quic_features[
    [
        "quic_version_id",
        "QUIC_TOKEN_LENGTH",
        "log_zero_rtt",
        "zero_rtt_present",
        "log_multiplexed",
        "multiplexed"
    ]
].copy()

selected_quic.head()

,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,1,70,0.693147,1,0.0,0
1,1,65,0.693147,1,0.0,0
2,1,0,0.000000,0,0.0,0
3,1,70,0.000000,0,0.0,0
4,1,0,0.000000,0,0.0,0


In [70]:
with_CID_df = pd.concat([ppi_final,cid_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(with_CID_df.shape)

with_CID_df.to_parquet(
    "with_CID.parquet",
    index=False
)

(728592, 40)


In [71]:
check_df = pd.read_parquet("with_CID.parquet")

print(check_df.shape)
check_df.head()

(728592, 40)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,occid_length,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,30,6,0.104360,1.540445,2.011873,435.633333,504.944649,0.413793,0.466667,0,...,7.377019,700.055556,0.158156,0.453474,1,70,0.693147,1,0.0,0
1,30,3,0.010940,0.312375,0.717000,629.766667,557.518351,0.172414,0.266667,0,...,13.720225,1157.797076,0.005772,0.102005,1,65,0.693147,1,0.0,0
2,30,6,0.810041,3.751854,5.377717,488.966667,517.350267,0.379310,0.433333,0,...,11.469826,1177.305864,0.008245,0.081001,1,0,0.000000,0,0.0,0
3,23,6,0.094401,1.668527,2.072350,568.173913,557.311220,0.500000,0.565217,0,...,11.876110,616.173913,1.671815,0.832909,1,70,0.000000,0,0.0,0
4,28,7,0.132781,1.803594,2.261974,408.500000,520.425478,0.481481,0.571429,0,...,11.365379,436.500000,0.546246,0.847298,1,0,0.000000,0,0.0,0


In [72]:
without_CID_df = pd.concat([ppi_final,hist_final,flow_features_selected,selected_quic],axis=1)

print(without_CID_df.shape)

without_CID_df.to_parquet(
    "without_CID.parquet",
    index=False
)

(728592, 26)


In [73]:
check_df2 = pd.read_parquet("without_CID.parquet")

print(check_df2.shape)
check_df2.head()

(728592, 26)


,PPI_LEN,PPI_ROUNDTRIPS,log_ppi_duration,log_mean_ipt,log_std_ipt,mean_packet_size,std_packet_size,direction_change_ratio,forward_packet_ratio,src_size_entropy,...,log_byte_rate,avg_packet_size,log_byte_ratio,log_packet_ratio,quic_version_id,QUIC_TOKEN_LENGTH,log_zero_rtt,zero_rtt_present,log_multiplexed,multiplexed
0,30,6,0.104360,1.540445,2.011873,435.633333,504.944649,0.413793,0.466667,0.673870,...,7.377019,700.055556,0.158156,0.453474,1,70,0.693147,1,0.0,0
1,30,3,0.010940,0.312375,0.717000,629.766667,557.518351,0.172414,0.266667,0.012406,...,13.720225,1157.797076,0.005772,0.102005,1,65,0.693147,1,0.0,0
2,30,6,0.810041,3.751854,5.377717,488.966667,517.350267,0.379310,0.433333,0.170845,...,11.469826,1177.305864,0.008245,0.081001,1,0,0.000000,0,0.0,0
3,23,6,0.094401,1.668527,2.072350,568.173913,557.311220,0.500000,0.565217,0.566813,...,11.876110,616.173913,1.671815,0.832909,1,70,0.000000,0,0.0,0
4,28,7,0.132781,1.803594,2.261974,408.500000,520.425478,0.481481,0.571429,0.598246,...,11.365379,436.500000,0.546246,0.847298,1,0,0.000000,0,0.0,0


In [74]:
print(with_CID_df.columns.duplicated().sum())
print(without_CID_df.columns.duplicated().sum())

0
0
